In [46]:
import pandas as pd

In [47]:
import numpy as np

In [48]:
control=pd.read_csv('/workspaces/a-b-testing-data-cleaning/notebooks/control_group.csv' , sep=';')

In [49]:
test = pd.read_csv('/workspaces/a-b-testing-data-cleaning/notebooks/test_group.csv' , sep=';')

In [50]:
control.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0
4,Control Campaign,5.08.2019,1835,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
test.head()

,Campaign Name,Date,Spend [USD],# of Impressions,Reach,# of Website Clicks,# of Searches,# of View Content,# of Add to Cart,# of Purchase
0,Test Campaign,1.08.2019,3008,39550,35820,3038,1946,1069,894,255
1,Test Campaign,2.08.2019,2542,100719,91236,4657,2359,1548,879,677
2,Test Campaign,3.08.2019,2365,70263,45198,7885,2572,2367,1268,578
3,Test Campaign,4.08.2019,2710,78451,25937,4216,2216,1437,566,340
4,Test Campaign,5.08.2019,2297,114295,95138,5863,2106,858,956,768


In [52]:
control['group'] = 'control'

test['group'] = 'test'

In [53]:
# merging the dataset
df = pd.concat([control, test], ignore_index=True)

In [54]:
df.shape


(60, 11)

In [55]:
#verifying 
df['group'].value_counts

<bound method IndexOpsMixin.value_counts of 0     control
1     control
2     control
3     control
4     control
5     control
6     control
7     control
8     control
9     control
10    control
11    control
12    control
13    control
14    control
15    control
16    control
17    control
18    control
19    control
20    control
21    control
22    control
23    control
24    control
25    control
26    control
27    control
28    control
29    control
30       test
31       test
32       test
33       test
34       test
35       test
36       test
37       test
38       test
39       test
40       test
41       test
42       test
43       test
44       test
45       test
46       test
47       test
48       test
49       test
50       test
51       test
52       test
53       test
54       test
55       test
56       test
57       test
58       test
59       test
Name: group, dtype: str>

In [56]:
# cleaning column names
df.columns = df.columns.str.strip()  
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('#', 'num')
df.columns = df.columns.str.replace('[', '')
df.columns = df.columns.str.replace(']', '')
df.columns = df.columns.str.replace('-', '_')

In [57]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Campaign_Name          60 non-null     str    
 1   Date                   60 non-null     str    
 2   Spend_USD              60 non-null     int64  
 3   num_of_Impressions     59 non-null     float64
 4   Reach                  59 non-null     float64
 5   num_of_Website_Clicks  59 non-null     float64
 6   num_of_Searches        59 non-null     float64
 7   num_of_View_Content    59 non-null     float64
 8   num_of_Add_to_Cart     59 non-null     float64
 9   num_of_Purchase        59 non-null     float64
 10  group                  60 non-null     str    
dtypes: float64(7), int64(1), str(3)
memory usage: 5.3 KB


In [58]:
df.isnull().sum()

Campaign_Name            0
Date                     0
Spend_USD                0
num_of_Impressions       1
Reach                    1
num_of_Website_Clicks    1
num_of_Searches          1
num_of_View_Content      1
num_of_Add_to_Cart       1
num_of_Purchase          1
group                    0
dtype: int64

In [59]:
df = df.dropna()

In [60]:
numeric_cols = [
    'Spend_USD',
    'num_of_Impressions',
    'Reach',
    'num_of_Website_Clicks',
    'num_of_Searches',
    'num_of_View_Content',
    'num_of_Add_to_Cart',
    'num_of_Purchase'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [61]:
df.groupby('group')[['num_of_Purchase', 'num_of_Website_Clicks']].sum()

,num_of_Purchase,num_of_Website_Clicks
group,,
control,15161.0,154303.0
test,15637.0,180970.0


In [62]:
# calculate conversion rates
control_cr = 15161 / 154303 *100
test_cr = 15637 / 180970 *100

control_cr, test_cr

(9.825473257162855, 8.640658672708183)

In [63]:
#calculating lift
lift = (test_cr - control_cr) / control_cr *100
lift

-12.058600674435018

In [64]:
df['conversion_rate'] = df['num_of_Purchase'] / df['num_of_Website_Clicks']

In [65]:
df[['group', 'conversion_rate']].head()

,group,conversion_rate
0,control,0.088084
1,control,0.063009
2,control,0.057160
3,control,0.110930
5,control,0.189672


In [66]:
df.groupby('group')['conversion_rate'].mean()

group
control    0.114772
test       0.092312
Name: conversion_rate, dtype: float64

In [67]:
control = 0.114772
test = 0.092312

lift = (test - control) / control
lift

-0.19569232913951132

In [68]:
from scipy.stats import ttest_ind

control_data = df[df['group']=='control']['conversion_rate']
test_data = df[df['group']=='test']['conversion_rate']

ttest_ind(control_data, test_data)

TtestResult(statistic=np.float64(1.500444703243478), pvalue=np.float64(0.1390184427183693), df=np.float64(57.0))

In [69]:
df['cost_per_purchase'] = df['Spend_USD'] / df['num_of_Purchase']

In [70]:
df.groupby('group')['cost_per_purchase'].mean()

group
control    5.052339
test       5.899589
Name: cost_per_purchase, dtype: float64

In [71]:
df['click_to_purchase'] = df['num_of_Purchase'] / df['num_of_Website_Clicks']
df['view_to_cart'] = df['num_of_Add_to_Cart'] / df['num_of_View_Content']

In [72]:
df.groupby('group')[['click_to_purchase','view_to_cart']].mean()

,click_to_purchase,view_to_cart
group,,
control,0.114772,0.777854
test,0.092312,0.515096


In [86]:
df.to_csv("finall_ab_analysis.csv", index=False)

In [73]:
df.head()

,Campaign_Name,Date,Spend_USD,num_of_Impressions,Reach,num_of_Website_Clicks,num_of_Searches,num_of_View_Content,num_of_Add_to_Cart,num_of_Purchase,group,conversion_rate,cost_per_purchase,click_to_purchase,view_to_cart
0,Control Campaign,1.08.2019,2280,82702.0,56930.0,7016.0,2290.0,2159.0,1819.0,618.0,control,0.088084,3.689320,0.088084,0.842520
1,Control Campaign,2.08.2019,1757,121040.0,102513.0,8110.0,2033.0,1841.0,1219.0,511.0,control,0.063009,3.438356,0.063009,0.662140
2,Control Campaign,3.08.2019,2343,131711.0,110862.0,6508.0,1737.0,1549.0,1134.0,372.0,control,0.057160,6.298387,0.057160,0.732085
3,Control Campaign,4.08.2019,1940,72878.0,61235.0,3065.0,1042.0,982.0,1183.0,340.0,control,0.110930,5.705882,0.110930,1.204684
5,Control Campaign,6.08.2019,3083,109076.0,87998.0,4028.0,1709.0,1249.0,784.0,764.0,control,0.189672,4.035340,0.189672,0.627702


In [74]:
df.columns

Index(['Campaign_Name', 'Date', 'Spend_USD', 'num_of_Impressions', 'Reach',
       'num_of_Website_Clicks', 'num_of_Searches', 'num_of_View_Content',
       'num_of_Add_to_Cart', 'num_of_Purchase', 'group', 'conversion_rate',
       'cost_per_purchase', 'click_to_purchase', 'view_to_cart'],
      dtype='str')

In [67]:
df.shape

(59, 15)

In [75]:
df.groupby(['Date','group'])['conversion_rate'].mean().unstack()

group,control,test
Date,,
1.08.2019,0.088084,0.083937
10.08.2019,0.322354,0.033846
11.08.2019,0.058375,0.178133
12.08.2019,0.265286,0.085794
13.08.2019,0.116875,0.107294
14.08.2019,0.174298,0.085149
15.08.2019,0.074755,0.079712
16.08.2019,0.083844,0.071618
17.08.2019,0.033494,0.030088


In [76]:
df.columns

Index(['Campaign_Name', 'Date', 'Spend_USD', 'num_of_Impressions', 'Reach',
       'num_of_Website_Clicks', 'num_of_Searches', 'num_of_View_Content',
       'num_of_Add_to_Cart', 'num_of_Purchase', 'group', 'conversion_rate',
       'cost_per_purchase', 'click_to_purchase', 'view_to_cart'],
      dtype='str')

In [39]:
df.columns = df.columns.str.strip()

In [77]:
for col in df.columns:
    print(col)

Campaign_Name
Date
Spend_USD
num_of_Impressions
Reach
num_of_Website_Clicks
num_of_Searches
num_of_View_Content
num_of_Add_to_Cart
num_of_Purchase
group
conversion_rate
cost_per_purchase
click_to_purchase
view_to_cart


In [78]:
df.columns = df.columns.str.strip().str.replace(' ', '_')

In [79]:
summary = df.groupby('group'). agg({
    'conversion_rate': 'mean',
    'cost_per_purchase': 'mean',
    'click_to_purchase': 'mean',
    'view_to_cart': 'mean',
    'num_of_Purchase': 'sum',
    'Spend_USD': 'sum'
}).round(4)

summary

,conversion_rate,cost_per_purchase,click_to_purchase,view_to_cart,num_of_Purchase,Spend_USD
group,,,,,,
control,0.1148,5.0523,0.1148,0.7779,15161.0,66818
test,0.0923,5.8996,0.0923,0.5151,15637.0,76892


In [80]:
df.groupby('group')[['conversion_rate','cost_per_purchase']].mean()

,conversion_rate,cost_per_purchase
group,,
control,0.114772,5.052339
test,0.092312,5.899589


In [85]:
df['Date'].head(10)

0    2019-08-01
1    2019-08-02
2    2019-08-03
3    2019-08-04
5    2019-08-06
6    2019-08-07
7    2019-08-08
8    2019-08-09
9    2019-08-10
10   2019-08-11
Name: Date, dtype: datetime64[us]

In [82]:
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y', errors='coerce')

In [83]:
df['Date'].isnull().sum()

np.int64(0)

In [84]:
df.info()

<class 'pandas.DataFrame'>
Index: 59 entries, 0 to 59
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Campaign_Name          59 non-null     str           
 1   Date                   59 non-null     datetime64[us]
 2   Spend_USD              59 non-null     int64         
 3   num_of_Impressions     59 non-null     float64       
 4   Reach                  59 non-null     float64       
 5   num_of_Website_Clicks  59 non-null     float64       
 6   num_of_Searches        59 non-null     float64       
 7   num_of_View_Content    59 non-null     float64       
 8   num_of_Add_to_Cart     59 non-null     float64       
 9   num_of_Purchase        59 non-null     float64       
 10  group                  59 non-null     str           
 11  conversion_rate        59 non-null     float64       
 12  cost_per_purchase      59 non-null     float64       
 13  click_to_purchase      